# Example 31 - Isotropic Bearings with Damping

This example is based on Example 5.9.5 from {cite}friswell2010dynamics

The isotropic bearing Example 5.9.1 (Example 17) is repeated but with damping in the bearings. The, _x_ and _y_ directions are uncoupled, with a translational stiffness of 1 MN/m and a damping of 3 kNs/m in each direction.

In [1]:
import ross as rs
import numpy as np
import pandas as pd

from ross.units import Q_
import plotly.io as pio

pio.renderers.default = "notebook_connected"

/home/raphaelts/ross-wt-release-3.2.0/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
steel = rs.Material("steel", E=211e9, G_s=81.2e9, rho=7810)

In [3]:
L = 0.25
N = 6
idl = 0
odl = 0.05

shaft = [rs.ShaftElement(L=L, idl=idl, odl=odl, material=steel) for i in range(N)]

bearings = [
    rs.BearingElement(n=0, kxx=1e6, kyy=1e6, cxx=3e3, cyy=3e3, scale_factor=2),
    rs.BearingElement(n=len(shaft), kxx=1e6, kyy=1e6, cxx=3e3, cyy=3e3, scale_factor=2),
]

disks = [
    rs.DiskElement.from_geometry(
        n=2, material=steel, width=0.07, i_d=odl, o_d=0.28, scale_factor="mass"
    ),
    rs.DiskElement.from_geometry(
        n=4, material=steel, width=0.07, i_d=odl, o_d=0.35, scale_factor="mass"
    ),
]

rotor = rs.Rotor(shaft_elements=shaft, disk_elements=disks, bearing_elements=bearings)
rotor.plot_rotor()

## Plotting the Campbell Diagram

In [4]:
campbell = rotor.run_campbell(
    speed_range=Q_(list(range(0, 4500, 50)), "RPM"), frequencies=8
)

In [5]:
campbell.plot(frequency_units="RPM")

## Plotting the Mode Shapes and Damped Natural Frequencies

In [6]:
modal = rotor.run_modal(speed=Q_(4000, "RPM"), num_modes=16)

In [7]:
for mode in range(7):
    display(modal.plot_mode_3d(mode, frequency_units="Hz"))

## Creating table with eigenvalues, natural frequencies and damping ratios

In [8]:
def modal_table(modal_results, rpm):
    eig = modal_results.evalues
    omega_n = np.abs(eig)
    omega_d = np.imag(eig)
    zeta = -np.real(eig) / omega_n
    fn = omega_n / (2 * np.pi)
    fd = omega_d / (2 * np.pi)

    return pd.DataFrame(
        {
            "Speed (RPM)": [rpm] * 8,
            "Root s (rad/s)": eig[:8],
            "Wn (Hz)": fn[:8],
            "Wd (Hz)": fd[:8],
            "Damping Ratio": zeta[:8],
        }
    )


modal_0 = rotor.run_modal(speed=Q_(0, "RPM"), num_modes=16)
modal_4000 = rotor.run_modal(speed=Q_(4000, "RPM"), num_modes=16)

df_modal_0 = modal_table(modal_0, rpm=0)
df_modal_4000 = modal_table(modal_4000, rpm=4000)
df_combined = pd.concat([df_modal_0, df_modal_4000], ignore_index=True)

display(df_combined)

,Speed (RPM),Root s (rad/s),Wn (Hz),Wd (Hz),Damping Ratio
0,0,-4.423835+ 87.257936j,13.905368,13.887532,5.063333e-02
1,0,-4.423835+ 87.257936j,13.905368,13.887532,5.063333e-02
2,0,-78.240826+292.422766j,48.177624,46.540529,2.584688e-01
3,0,-78.240826+292.422766j,48.177624,46.540529,2.584688e-01
4,0,-566.540378+648.581404j,137.060578,103.224936,6.578675e-01
5,0,-566.540378+648.581404j,137.060578,103.224936,6.578675e-01
6,0,0.000000+650.407866j,103.515627,103.515627,-2.814811e-13
7,0,-657.308076+834.757536j,169.099715,132.855788,6.186517e-01
8,4000,-4.083081+ 85.965810j,13.697307,13.681884,4.744309e-02
9,4000,-4.741740+ 88.414706j,14.091860,14.071637,5.355371e-02
